# 🐳 PHASE 2: Docker, Netzwerk & Reverse Proxy (Traefik + OAuth2-Proxy)
**Projekt: Mai_AI (MaiOmni) — Reverse Proxy & Dynamic routing**

In dieser Phase schalten wir das schützende **Traefik-Gateway** und den **Google OAuth2 Proxy** vor dein System. Dies entspricht der Architektur-Spezifikation für einen sicheren Web-Zugang über DuckDNS (`mai-ai.duckdns.org`), ohne den Host angreifbar zu machen.

--- 
### 🎯 Ziele dieser Phase:
1. Validierung des lokalen Docker-Status.
2. Erstellung des isolierten Plattform-Netzwerks (`ai_platform_network`).
3. Initialisierung von Traefik und des Auth-Gateways über Docker-Compose.
4. Verifizierung des sicheren Header-Verhaltens (`X-Forwarded-User`).

---

### 🛠️ Schritt 1: Docker CLI & Daemon Konnektivität prüfen
Wir prüfen mittels Python Docker SDK, ob die Docker Engine läuft und erreichbar ist.

In [1]:
import docker
import os
import sys
import time
import subprocess

def connect_docker(retries=3, delay=5):
    """
    Versucht eine Verbindung zum Docker-Daemon herzustellen.
    Falls Docker nicht läuft, wird versucht, die Anwendung automatisch zu starten.
    Unter macOS geschieht dies geräuschlos im Hintergrund.
    """
    for i in range(retries):
        try:
            # Verbindung herstellen und prüfen
            client = docker.from_env()
            client.ping()
            print(f"[✓] Docker-Daemon ist aktiv. Server Version: {client.version()['Version']}")
            return client
        except (docker.errors.DockerException, Exception) as e:
            print(f"[!] Docker ist offline oder nicht erreichbar (Versuch {i+1}/{retries}).")
            
            if i < retries - 1:
                print(f"[...] Versuche Docker automatisch zu starten... Bitte warten.")
                try:
                    # Automatischer Start je nach Betriebssystem
                    if sys.platform == "darwin":  # macOS
                        # -g hält das GUI-Fenster im Hintergrund geschlossen
                        subprocess.run(["open", "-g", "-a", "Docker"], check=True)
                    elif sys.platform == "win32":  # Windows
                        # Standardpfad für Docker Desktop unter Windows
                        docker_path = r"C:\Program Files\Docker\Docker\Docker Desktop.exe"
                        if os.path.exists(docker_path):
                            subprocess.Popen([docker_path], start_new_session=True)
                        else:
                            print("[X] Docker Desktop Pfad unter Windows nicht gefunden.")
                    elif sys.platform.startswith("linux"):  # Linux
                        subprocess.run(["sudo", "systemctl", "start", "docker"], check=True)
                except Exception as start_error:
                    print(f"[X] Fehler beim Startversuch von Docker: {start_error}")
                
                # Wartezeit, damit der Daemon Zeit zum Hochfahren hat
                time.sleep(delay)
            else:
                print("\n[X] Docker konnte nicht gestartet werden. Bitte öffne Docker Desktop manuell.")
                raise e

# Skript ausführen
try:
    client = connect_docker()
except Exception as final_error:
    print(f"\nAbbruch: Verbindung zu Docker fehlgeschlagen.\nDetails: {final_error}")

[!] Docker ist offline oder nicht erreichbar (Versuch 1/3).
[...] Versuche Docker automatisch zu starten... Bitte warten.
[✓] Docker-Daemon ist aktiv. Server Version: 29.5.2


### 🌐 Schritt 2: Docker-Netzwerk initialisieren
Die dynamic user containers kommunizieren über das dedizierte Brücken-Netzwerk `ai_platform_network` mit Traefik.

In [ ]:
network_name = "ai_platform_network"
try:
    networks = client.networks.list(names=[network_name])
    if not networks:
        client.networks.create(network_name, driver="bridge", attachable=True)
        print(f"[✓] Netzwerk '{network_name}' wurde erfolgreich erstellt.")
    else:
        print(f"[✓] Netzwerk '{network_name}' ist bereits vorhanden.")
except Exception as e:
    print(f"[!] Fehler beim Erstellen des Netzwerks: {e}")

### 🏗️ Schritt 3: Infrastruktur-Gateway starten (Traefik & OAuth2-Proxy)
Jetzt rufen wir die systemeigene Start-Sequenz auf, um Traefik und Google OAuth2 Proxy hochzufahren. (Stelle sicher, dass du eine `.env` Datei basierend auf `.env.example` erstellt hast!)

In [ ]:
import os
import subprocess

if not os.path.exists("../.env"):
    print("[!] Warnung: Keine .env Datei gefunden! Kopiere '.env.example' zu '.env' und trag deine Daten ein.")
else:
    print("[✓] .env Datei gefunden. Starte Docker Compose...")
    # Führt das Start-AI Skript aus, um die Docker-Infrastruktur hochzufahren
    try:
        subprocess.run(["python", "../src/docker_py/Start_AI.py"], check=True)
    except Exception as e:
        print(f"[!] Fehler beim Startvorgang: {e}")

### 🔄 Was kommt als Nächstes?
Deine Gateway-Infrastruktur läuft im Hintergrund und filtert unautorisierte Anfragen.

Fahre fort mit dem nächsten Knotenpunkt: 
👉 **[03_html_embed.ipynb](file:notebooks/03_html_embed.ipynb)** um das Streamlit-User-Image zu bauen und den Dynamic Provisioner zu testen.